# TILDE (2021)
---
[[paper]](https://arxiv.org/abs/2104.14081) <br>
TILDE = Term Independent Likelihood moDEl

TILDE — это метод Sparse Retrieval (разреженного поиска), предложенный исследователями из University of Glasgow. Он использует предобученные языковые модели (BERT) для предсказания весов всех токенов из словаря для каждого документа, что позволяет эффективно решать проблему Vocabulary Mismatch при сохранении скорости классического поиска.

**Идея:** вместо того чтобы кодировать документ в плотный вектор (как в DPR) или использовать сырые частоты слов (как в BM25), давайте для каждого документа заранее предскажем вероятность того, что конкретное слово из всего словаря BERT появится в релевантном запросе к этому документу. 

**Задача:** Information Retrieval. Из большого корпуса документов $D$ нужно найти наиболее релевантные для запроса $q$, минимизируя вычислительные затраты на этапе инференса.

### Альтернативные методы на момент появления TILDE
*   **BM25 (1990-е):** использует статистику частот слов. Главная проблема — Vocabulary Mismatch (если в запросе "автомобиль", а в документе "машина", BM25 их не свяжет).
*   **DeepCT (2019):** использует BERT для предсказания новых весов (вместо TF) для слов, *уже присутствующих* в документе. Не решает проблему расширения словаря.
*   **Doc2Query (2019):** использует seq2seq модель (T5) для генерации гипотетических запросов к документу и дописывает их в конец текста. Это расширяет документ, но процесс генерации очень медленный и стохастичный.
*   **DPR (2020):** Dense Retrieval на базе двух башен BERT. Требует больших мощностей для хранения векторного индекса и вычисления сходства (Nearest Neighbor Search).

### Архитектура
TILDE построена на базе BERT, но используется специфическим образом:
1.  **Encoder:** Стандартный BERT (checkpoint `bert-base-uncased`).
2.  **Prediction Head:** Линейный слой поверх выходных эмбеддингов BERT, который проецирует их в пространство размерности $|V|$, где $|V|$ — размер словаря (30 522 токена).
3.  **Aggregation:** Для документа из $n$ токенов после прохода через BERT получается матрица $n \times |V|$. Метод агрегирует её (например, через усреднение), получая один вектор весов размерности словаря для всего документа.

### Алгоритм обучения
Модель обучается предсказывать вероятность появления терма в запросе, основываясь на содержании релевантного документа.
*   В качестве обучающего сета используется MS MARCO (пары запрос-релевантный пассаж).
*   Для каждого пассажа из обучающей выборки модель должна максимизировать вероятность (Log-Likelihood) тех токенов, которые реально присутствуют в исходном запросе пользователя.
*   **Loss function:** Binary Cross Entropy для каждого токена из словаря. Модель учится бинарной классификации: "появится ли этот токен из словаря в запросе к данному документу или нет".

### Алгоритм инференса
Ключевое преимущество TILDE — **Query-free encoding**. На этапе обработки запроса нейросеть вообще не нужна.

**Этап индексации (Offline):**
1.  Каждый документ пропускается через TILDE.
2.  Модель генерирует веса $w$ для каждого слова из словаря BERT. Большая часть весов будет близка к нулю (используется порог отсечения, например, топ-200 слов или веса > threshold).
3.  Создается **Inverted Index**, аналогичный индексу в Lucene/Elasticsearch, где вместо частоты слова (TF) хранятся предсказанные веса TILDE.

**Этап поиска (Online):**
1.  Запрос токенизируется.
2.  Score документа вычисляется как простая сумма весов токенов запроса, взятых из индекса: $Score(q, d) = \sum_{t \in q} w_{t,d}$.
3.  Это делает инференс сопоставимым по скорости с BM25, так как это просто выборка из хэш-таблицы и суммирование нескольких чисел.

### Новизна подхода
В отличие от методов Dense Retrieval, TILDE не пытается сжать смысл документа в один вектор. Вместо этого она выполняет **Expansion** (расширение) документа на весь словарь BERT. Новизна заключается в **Term Independence**: веса термов в индексе не зависят от других термов в будущем запросе. Это позволяет перенести все тяжелые нейросетевые вычисления на этап оффлайн-индексации.

### Результаты
*   **Скорость:** TILDE работает в **40-50 раз быстрее**, чем стандартные нейросетевые Rerankers (Cross-Encoders), и значительно быстрее Dense Retrieval моделей, так как не требует векторного поиска.
*   **Качество:** На датасете MS MARCO (Passage Retrieval) TILDE показала результат MRR@10 около 0.30, что значительно выше, чем у BM25 (0.19), и сопоставимо с первыми версиями DPR, при этом используя на порядки меньше памяти для индекса.
*   **Эффективность памяти:** За счет использования разреженного индекса (Sparse Index) вместо плотных векторов, объем памяти для хранения данных сокращается в несколько раз по сравнению с Faiss-индексами для Dense моделей.

## 📝 Критический анализ

```markdown
# TILDE (2021)
---
[[paper]](https://arxiv.org/abs/2104.14081)  
TILDE = Term Independent Likelihood moDEl

TILDE — метод Sparse Retrieval, разработанный в University of Glasgow. Он использует BERT для предсказания весов токенов из словаря для каждого документа, решая проблему Vocabulary Mismatch и сохраняя скорость классического поиска.

**Идея:** Вместо плотного вектора или частот слов, предсказываем вероятность появления каждого слова из словаря BERT в релевантном запросе к документу.

**Задача:** Information Retrieval — найти релевантные документы для запроса $q$ с минимальными вычислительными затратами на инференс.

### Альтернативы
* **BM25 (1990-е):** Использует частоты слов, страдает от Vocabulary Mismatch.
* **DeepCT (2019):** Прогнозирует веса для существующих слов, не расширяет словарь.
* **Doc2Query (2019):** Генерирует гипотетические запросы, но медленно.
* **DPR (2020):** Требует больших мощностей для векторного поиска.

### Архитектура
1. **Encoder:** BERT (`bert-base-uncased`).
2. **Prediction Head:** Линейный слой, проецирующий в пространство размерности $|V|$.
3. **Aggregation:** Агрегация матрицы $n \times |V|$ в вектор весов.

### Алгоритм обучения
Обучается на MS MARCO, максимизируя вероятность токенов из запросов. Используется Binary Cross Entropy для классификации токенов.

### Алгоритм инференса
**Индексация (Offline):**
1. Пропуск документов через TILDE.
2. Генерация весов $w$ для слов из словаря BERT.
3. Создание Inverted Index с предсказанными весами.

**Поиск (Online):**
1. Токенизация запроса.
2. Вычисление Score как суммы весов токенов запроса.

<img src="img/img.png" width=500>

### Новизна
TILDE не сжимает смысл в один вектор, а расширяет документ на весь словарь BERT. **Term Independence** позволяет перенести вычисления на оффлайн-индексацию.

### Результаты
* **Скорость:** В 40-50 раз быстрее стандартных Rerankers и Dense Retrieval.
* **Качество:** MRR@10 на MS MARCO около 0.30, выше BM25 (0.19), сопоставимо с DPR.
* **Эффективность памяти:** Использует Sparse Index, сокращая объем памяти по сравнению с Faiss-индексами.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import numpy as np

# Загрузка предобученной модели BERT и токенизатора
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Определение линейного слоя для предсказания весов токенов
class TILDE(nn.Module):
    def __init__(self, vocab_size):
        super(TILDE, self).__init__()
        self.bert = model
        self.linear = nn.Linear(self.bert.config.hidden_size, vocab_size)

    def forward(self, input_ids, attention_mask):
        # Получение эмбеддингов из BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Применение линейного слоя для проекции в пространство размерности словаря
        logits = self.linear(outputs.last_hidden_state)
        # Усреднение по токенам для получения одного вектора весов на документ
        weights = logits.mean(dim=1)
        return weights

# Инициализация модели TILDE
vocab_size = len(tokenizer)
tilde_model = TILDE(vocab_size)

# Пример документа
document = "The quick brown fox jumps over the lazy dog."

# Токенизация документа
inputs = tokenizer(document, return_tensors='pt', truncation=True, padding=True)

# Получение весов токенов для документа
with torch.no_grad():
    weights = tilde_model(inputs['input_ids'], inputs['attention_mask'])

# Пример индексации: создание разреженного индекса
# Здесь мы сохраняем только топ-5 токенов с наибольшими весами
top_k = 5
top_weights, top_indices = torch.topk(weights, top_k, dim=1)
inverted_index = {tokenizer.convert_ids_to_tokens(idx.item()): weight.item() 
                  for idx, weight in zip(top_indices[0], top_weights[0])}

print("Inverted Index for the document:")
print(inverted_index)

# Пример поиска: вычисление score для запроса
query = "quick fox"
query_tokens = tokenizer.tokenize(query)

# Суммирование весов токенов запроса из индекса
score = sum(inverted_index.get(token, 0) for token in query_tokens)
print(f"Score for query '{query}': {score}")
```

### Объяснение ключевых моментов:

1. **Модель TILDE:** Мы используем предобученную модель BERT и добавляем линейный слой, чтобы проецировать выходные эмбеддинги BERT в пространство размерности словаря. Это позволяет нам предсказывать веса для всех токенов словаря.

2. **Агрегация:** После получения матрицы весов размерности `n x |V|` (где `n` — количество токенов в документе), мы усредняем её по токенам, чтобы получить один вектор весов для всего документа.

3. **Индексация:** Мы создаем разреженный индекс, сохраняя только токены с наибольшими весами. Это позволяет эффективно хранить и обрабатывать данные.

4. **Поиск:** На этапе поиска мы просто суммируем веса токенов запроса, извлеченные из индекса. Это делает инференс быстрым и эффективным, аналогично классическим методам, таким как BM25.

Этот пример иллюстрирует, как TILDE решает проблему Vocabulary Mismatch, предсказывая вероятности появления токенов в запросах, и как это позволяет эффективно выполнять поиск без необходимости в плотных векторах.